# 04 - Particionamiento y Adaptive Query Execution (AQE)

### Diferencia técnica entre operaciones
* **`repartition(n)`**: Realiza un shuffle completo en la red. Útil para balancear particiones sesgadas.
* **`coalesce(n)`**: Reduce particiones combinando bloques locales sin generar shuffle.


In [1]:
import sys
sys.path.append("..")
from src.config import get_spark_session
import pyspark.sql.functions as F

spark = get_spark_session("04_AQE")
df_num = spark.range(0, 500000, 1, numPartitions=8)
print("Particiones iniciales :", df_num.rdd.getNumPartitions())
print("Tras coalesce(2)      :", df_num.coalesce(2).rdd.getNumPartitions())
print("Tras repartition(10)  :", df_num.repartition(10).rdd.getNumPartitions())


Particiones iniciales : 8
Tras coalesce(2)      : 2
Tras repartition(10)  : 10


### Ajuste Dinámico con AQE
AQE coalescee particiones intermedias automáticamente al detectar etapas con volúmenes bajos:


In [2]:
df_agg = df_num.groupBy(F.col("id") % 4).count()
df_agg.explain(mode="cost")
df_agg.show()


== Optimized Logical Plan ==
Aggregate [_groupingexpression#12L], [_groupingexpression#12L AS (id % 4)#9L, count(1) AS count#8L], Statistics(sizeInBytes=5.7 MiB)
+- Project [(id#0L % 4) AS _groupingexpression#12L], Statistics(sizeInBytes=3.8 MiB)
   +- Range (0, 500000, step=1, splits=Some(8)), Statistics(sizeInBytes=3.8 MiB, rowCount=5.00E+5)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[_groupingexpression#12L], functions=[count(1)], output=[(id % 4)#9L, count#8L])
   +- Exchange hashpartitioning(_groupingexpression#12L, 4), ENSURE_REQUIREMENTS, [plan_id=44]
      +- HashAggregate(keys=[_groupingexpression#12L], functions=[partial_count(1)], output=[_groupingexpression#12L, count#14L])
         +- Project [(id#0L % 4) AS _groupingexpression#12L]
            +- Range (0, 500000, step=1, splits=8)


+--------+------+
|(id % 4)| count|
+--------+------+
|       2|125000|
|       0|125000|
|       1|125000|
|       3|125000|
+--------+------+

